# Narrow vs Wide Transformations: Shuffles and Stage Boundaries

**Module**: 04_execution_engine  
**Notebook**: 06_narrow_and_wide_transformations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Categorizing transformations as Narrow (map, filter) or Wide (groupBy, join, distinct, repartition) and observing network shuffles.
- **Why does this topic exist?**: Wide transformations dictate execution time and network I/O in distributed systems.
- **What problem does it solve?**: Unnecessary wide transformations destroy query scalability.
- **Where is it used in real-world Data Engineering?**: ETL pipeline performance optimization.

---

## 2. Narrow (1:1 partition) vs Wide (1:N or N:M shuffle)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Narrow_Wide_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("=== Narrow Plan (No Exchange) ===")
orders.filter("quantity > 2").select("order_id").explain()

print("\n=== Wide Plan (Contains Exchange) ===")
orders.groupBy("status").count().explain()

---

## Experiment: Exchange Node Detection

- **1. What are we testing?**: Look for Exchange in physical plan.
- **2. What do we expect?**: Present only in wide transformations.

In [ ]:
plan = orders.distinct()._jdf.queryExecution().executedPlan().toString(); print('Contains Exchange?:', 'Exchange' in plan)

**3. What happened?**: Exchange node present.  
**4. Why did it happen?**: distinct() requires a shuffle to deduplicate across partitions.  
**5. Practical Engineering Lesson**: Exchange in physical plan = network shuffle = stage boundary.

---

## Practical Exercises

### Exercise 1 (Beginner): Classify Filter
**Problem Statement**: Is filter narrow or wide?  
**Input / Context**: filter

In [ ]:
# B

### Exercise 2 (Intermediate): Classify Distinct
**Problem Statement**: Is distinct narrow or wide?  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Pipeline Optimization
**Problem Statement**: Filter before wide groupBy.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Classify Filter):
```python
print('Narrow')
```

#### Solution 2 (Classify Distinct):
```python
print('Wide (Exchange node)')
```

#### Solution 3 (Pipeline Optimization):
```python
orders.filter('quantity > 5').groupBy('channel').count().show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Narrow**: In-memory pipelining, no network transfer.
- **Wide**: Network shuffle, writes shuffle files to disk.

### Essential Technical Terms
- `Exchange`: Physical operator executing shuffle.
- `Pipelining`: Chaining narrow steps.

### Common Engineering Mistakes
- **Assuming distinct() is free**: distinct() requires a full network shuffle.

### Interview Questions on This Topic
- **Q: Why is narrow faster than wide?**
  - *A*: Narrow does not write to disk or transfer data over the network.

### What to Study Next
Open 07_dependencies_and_stage_boundaries.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')